# Train the image captioner on Kaggle

**Before running:**
1. Session options → Accelerator → **GPU** (T4 or P100), **Internet on**, Persistence → **Files only**.
2. Add Input → the **COCO2014** dataset (folders `captions/annotations`, `train2014/train2014`, `val2014/val2014`).
3. Paste that dataset's path into `COCO_ROOT` below (copy icon next to the dataset name in the Input panel).

In [ ]:
COCO_ROOT = "/kaggle/input/coco2014"     # <- paste your COCO2014 path here
KARPATHY_JSON = None                       # optional: path to dataset_coco.json if you add one

import os
TRAIN_IMAGES = f"{COCO_ROOT}/train2014/train2014"
VAL_IMAGES = f"{COCO_ROOT}/val2014/val2014"
CAPTIONS = f"{COCO_ROOT}/captions/annotations"
print("train images:", len(os.listdir(TRAIN_IMAGES)))   # expect 82,783
print("val images:", len(os.listdir(VAL_IMAGES)))       # expect 40,504
print(os.listdir(CAPTIONS))

In [ ]:
!git clone https://github.com/PiyushaKadam07/Image-Captioning-Transformer.git
%cd Image-Captioning-Transformer
!pip install -q -r requirements.txt

## 1. Prepare the splits

In [ ]:
if KARPATHY_JSON:
    !python scripts/prepare_data.py --karpathy {KARPATHY_JSON} --out data
else:
    !python scripts/prepare_data.py --coco-train {CAPTIONS}/captions_train2014.json --coco-val {CAPTIONS}/captions_val2014.json --out data
!mkdir -p data/images
!ln -sfn {TRAIN_IMAGES} data/images/train2014
!ln -sfn {VAL_IMAGES} data/images/val2014
!head -c 300 data/test.jsonl

## 2. Quick check (about 5 minutes)
Trains on 5,000 captions for one epoch to make sure everything works before the long runs.

In [ ]:
!python scripts/train.py --config configs/resnet50.yaml --subset 5000 --epochs 1 --output-dir runs/check

## 3. Full training
Each run takes several hours on a free GPU. Kaggle sessions stop after about 12 hours and allow limited GPU hours per week, so run one encoder per session. If a session ends, `last.pt` and `log.csv` in the output folder show how far it got.
To keep the checkpoints, **Save Version → Save & Run All** so `/kaggle/working` is kept as output.

In [ ]:
!python scripts/train.py --config configs/resnet50.yaml --output-dir /kaggle/working/runs/resnet50

In [ ]:
!python scripts/train.py --config configs/clip.yaml --output-dir /kaggle/working/runs/clip

## 4. Evaluate on the 5,000-image test split (beam search and greedy, for the ablation table)

In [ ]:
for enc in ["resnet50", "clip"]:
    ckpt = f"/kaggle/working/runs/{enc}/best.pt"
    !python scripts/evaluate.py --checkpoint {ckpt} --beam-size 3
    !python scripts/evaluate.py --checkpoint {ckpt} --beam-size 1
!python scripts/results_table.py /kaggle/working/runs

## 5. Try it on an image

In [ ]:
import glob
from IPython.display import Image as Show
sample = sorted(glob.glob(f"{VAL_IMAGES}/*.jpg"))[0]
!python scripts/caption.py --checkpoint /kaggle/working/runs/clip/best.pt --image {sample} --attention attention.png
Show("attention.png")